## Instructions for Running the Demo

To run this DeepSeek + GDELT integration demo:

### 1. Start the OpenLLM Server

Open a terminal and run:
```bash
cd ex-accell-ui
poetry run openllm serve deepseek:r1-llama3.1-8b --port 3000
```

Wait for the server to start (you'll see "Server is ready" message).

### 2. Run the Notebook Cells

Execute each cell in sequence. The notebook will:

1. **Setup**: Check connections to DeepSeek and GDELT database
2. **Demo 1**: Analyze recent global events with geopolitical insights
3. **Demo 2**: Focus on a specific country (US by default)
4. **Demo 3**: Analyze trends and provide predictions
5. **Demo 4**: Interactive analysis where you can ask custom questions

### 3. Expected Results

- **Geopolitical Analysis**: DeepSeek provides detailed analysis of global events
- **Country-Specific Insights**: Focused analysis on national activities and relationships
- **Trend Analysis**: Pattern recognition and future predictions
- **Interactive Q&A**: Real-time answers to your questions about the data

### 4. Performance Notes

- **Model Loading**: First query may take 1-2 minutes
- **Subsequent Queries**: 30-60 seconds per response
- **GPU Requirements**: DeepSeek R1 8B model needs ~24GB GPU RAM
- **CPU Fallback**: Will work on CPU but significantly slower

### 5. Integration Points

This demo showcases how to:
- Combine AI reasoning with real-time data
- Generate actionable intelligence from raw events
- Provide context-aware analysis
- Create interactive data exploration tools

The integration can be extended to:
- Generate automated reports
- Create early warning systems
- Provide real-time briefings
- Support decision-making processes

In [ ]:
# Interactive GDELT analysis with DeepSeek
def interactive_gdelt_analysis():
    """Interactive function to ask questions about GDELT data"""
    
    def get_custom_data(query_description):
        """Get data based on user description"""
        try:
            conn = duckdb.connect(GDELT_DB_PATH)
            
            # For demo, we'll provide some common query patterns
            if "conflict" in query_description.lower() or "negative" in query_description.lower():
                query = """
                SELECT Actor1Name, Actor2Name, EventCode, GoldsteinScale, AvgTone, 
                       ActionGeo_Fullname, ActionGeo_CountryCode
                FROM gdelt_events 
                WHERE GoldsteinScale < -5 AND AvgTone < -5
                ORDER BY GoldsteinScale ASC, AvgTone ASC
                LIMIT 10
                """
            elif "cooperation" in query_description.lower() or "positive" in query_description.lower():
                query = """
                SELECT Actor1Name, Actor2Name, EventCode, GoldsteinScale, AvgTone, 
                       ActionGeo_Fullname, ActionGeo_CountryCode
                FROM gdelt_events 
                WHERE GoldsteinScale > 5 AND AvgTone > 5
                ORDER BY GoldsteinScale DESC, AvgTone DESC
                LIMIT 10
                """
            else:
                # Default: recent high-impact events
                query = """
                SELECT Actor1Name, Actor2Name, EventCode, GoldsteinScale, AvgTone, 
                       ActionGeo_Fullname, ActionGeo_CountryCode
                FROM gdelt_events 
                WHERE SQLDATE >= 20250120
                ORDER BY ABS(GoldsteinScale) DESC
                LIMIT 10
                """
            
            df = conn.execute(query).df()
            conn.close()
            return df
        except Exception as e:
            print(f"Error fetching custom data: {e}")
            return pd.DataFrame()
    
    print("🔍 INTERACTIVE GDELT ANALYSIS WITH DEEPSEEK")
    print("="*60)
    print("Ask me questions about global events! Type 'quit' to exit.")
    print("\nExample questions:")
    print("- 'What conflicts are happening recently?'")
    print("- 'Show me positive cooperation events'")
    print("- 'What's happening with China?'")
    print("- 'Analyze recent diplomatic activities'")
    print()
    
    while True:
        try:
            user_question = input("\n🤔 Your question: ").strip()
            
            if user_question.lower() in ['quit', 'exit', 'bye', 'stop']:
                print("👋 Goodbye!")
                break
                
            if not user_question:
                continue
            
            print(f"\n🔄 Analyzing your question: '{user_question}'")
            
            # Get relevant data
            relevant_data = get_custom_data(user_question)
            
            if relevant_data.empty:
                print("❌ No relevant data found for your question.")
                continue
            
            # Format data for DeepSeek
            data_summary = f"Relevant GDELT events for: '{user_question}'\\n\\n"
            for i, (_, row) in enumerate(relevant_data.iterrows(), 1):
                data_summary += f"{i}. {row['Actor1Name']} -> {row['Actor2Name']} "
                data_summary += f"(Event: {row['EventCode']}, Goldstein: {row['GoldsteinScale']}, "
                data_summary += f"Tone: {row['AvgTone']:.1f}, Location: {row['ActionGeo_Fullname']})\\n"
            
            # Create analysis prompt
            analysis_prompt = f'''
            User Question: "{user_question}"
            
            Relevant GDELT Data:
            {data_summary}
            
            Please provide a comprehensive answer to the user's question based on this GDELT data. 
            Be specific, cite the events, and provide actionable insights. 
            If the question asks for analysis, provide detailed geopolitical context.
            '''
            
            if not check_deepseek_server():
                print("❌ DeepSeek server not available. Please start the server first.")
                continue
            
            print("🤖 DeepSeek is analyzing...")
            
            start_time = time.time()
            response = query_deepseek(analysis_prompt, max_tokens=1500, temperature=0.5)
            end_time = time.time()
            
            if "error" in response:
                print(f"❌ Error: {response['error']}")
                continue
            
            answer = response['choices'][0]['message']['content']
            print("\\n🎯 DEEPSEEK ANALYSIS:")
            print("-" * 50)
            print(answer)
            print("-" * 50)
            print(f"⏱️ Response time: {end_time - start_time:.1f} seconds")
            
        except KeyboardInterrupt:
            print("\\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {e}")

# Example questions to try (uncomment to use interactively)
print("📝 SAMPLE QUESTIONS TO TRY:")
print("="*50)
sample_questions = [
    "What conflicts are happening recently?",
    "Show me positive cooperation events",
    "Analyze diplomatic activities",
    "What events show negative sentiment?",
    "What are the most significant recent events?"
]

for i, question in enumerate(sample_questions, 1):
    print(f"{i}. {question}")

print("\\n💡 To start interactive analysis, uncomment the line below:")
print("# interactive_gdelt_analysis()")

# Uncomment the next line to start interactive mode
# interactive_gdelt_analysis()

## Demo 4: Interactive Analysis

Create an interactive function where you can ask DeepSeek specific questions about the GDELT data.

In [ ]:
# Ask DeepSeek for trend analysis and predictions
if trend_summary and check_deepseek_server():
    prediction_prompt = f"""
    As a geopolitical forecasting analyst, please analyze these global events trends and provide insights:

    {trend_summary}

    Please provide:
    1. Key trends and patterns you observe in the data
    2. Countries or regions showing concerning developments
    3. Areas of increased cooperation vs. conflict
    4. Sentiment shifts over time (based on tone analysis)
    5. Potential early warning signals or indicators to monitor
    6. Short-term predictions (next 1-2 weeks) based on these patterns
    7. Recommended focus areas for continued monitoring
    8. Any seasonal or cyclical patterns you notice

    Be specific about what the data suggests and provide actionable intelligence insights.
    Focus on identifying potential escalations, de-escalations, or significant diplomatic developments.
    """
    
    print("🤖 Asking DeepSeek for trend analysis and predictions...")
    print("This may take 60-90 seconds for comprehensive analysis...\n")
    
    start_time = time.time()
    response = query_deepseek(prediction_prompt, max_tokens=2500, temperature=0.4)
    end_time = time.time()
    
    if "error" in response:
        print(f"❌ Error: {response['error']}")
    else:
        prediction_analysis = response['choices'][0]['message']['content']
        print("🔮 DEEPSEEK TREND ANALYSIS & PREDICTIONS:")
        print("="*80)
        print(prediction_analysis)
        print("="*80)
        print(f"\n⏱️ Analysis completed in {end_time - start_time:.1f} seconds")
        
else:
    print("⚠️ Skipping trend analysis - either no data or server not available")

In [ ]:
# Analyze trends over time
def get_trend_data(days_back=30):
    """Get aggregated data for trend analysis"""
    try:
        conn = duckdb.connect(GDELT_DB_PATH)
        
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days_back)
        start_date_str = start_date.strftime("%Y%m%d")
        
        query = """
        SELECT 
            SQLDATE,
            COUNT(*) as event_count,
            AVG(GoldsteinScale) as avg_goldstein,
            AVG(AvgTone) as avg_tone,
            COUNT(DISTINCT ActionGeo_CountryCode) as countries_involved,
            ActionGeo_CountryCode as country,
            COUNT(*) as country_events
        FROM gdelt_events 
        WHERE SQLDATE >= ?
        AND GoldsteinScale IS NOT NULL
        AND AvgTone IS NOT NULL
        GROUP BY SQLDATE, ActionGeo_CountryCode
        ORDER BY SQLDATE DESC, country_events DESC
        LIMIT 100
        """
        
        df = conn.execute(query, [start_date_str]).df()
        conn.close()
        return df
    except Exception as e:
        print(f"Error fetching trend data: {e}")
        return pd.DataFrame()

print("📈 Fetching trend data for the last 30 days...")
trend_data = get_trend_data(30)

if not trend_data.empty:
    print(f"✅ Found trend data for {len(trend_data)} country-day combinations")
    
    # Aggregate by date
    daily_trends = trend_data.groupby('SQLDATE').agg({
        'event_count': 'sum',
        'avg_goldstein': 'mean',
        'avg_tone': 'mean',
        'country': 'count'  # This gives us countries per day
    }).reset_index()
    
    print("\n📊 DAILY TRENDS SUMMARY:")
    print("="*60)
    for _, row in daily_trends.head(10).iterrows():
        date_str = str(row['SQLDATE'])
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        print(f"{formatted_date}: {row['event_count']} events, "
              f"Goldstein: {row['avg_goldstein']:.2f}, "
              f"Tone: {row['avg_tone']:.2f}, "
              f"Countries: {row['country']}")
    
    # Get most active countries
    country_activity = trend_data.groupby('country').agg({
        'country_events': 'sum',
        'avg_goldstein': 'mean',
        'avg_tone': 'mean'
    }).sort_values('country_events', ascending=False).head(10)
    
    print(f"\n🌍 TOP 10 MOST ACTIVE COUNTRIES:")
    print("="*60)
    for country, data in country_activity.iterrows():
        if pd.notna(country):  # Skip null countries
            print(f"{country}: {data['country_events']} events, "
                  f"Avg Goldstein: {data['avg_goldstein']:.2f}, "
                  f"Avg Tone: {data['avg_tone']:.2f}")
    
    # Prepare trend summary for DeepSeek
    trend_summary = f"""
GLOBAL EVENTS TREND ANALYSIS (Last 30 days):

Daily Activity Patterns:
"""
    for _, row in daily_trends.head(7).iterrows():
        date_str = str(row['SQLDATE'])
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        trend_summary += f"- {formatted_date}: {row['event_count']} events (Goldstein: {row['avg_goldstein']:.2f}, Tone: {row['avg_tone']:.2f})\n"
    
    trend_summary += f"\nMost Active Countries (by event count):\n"
    for country, data in country_activity.head(5).iterrows():
        if pd.notna(country):
            trend_summary += f"- {country}: {data['country_events']} events (Goldstein: {data['avg_goldstein']:.2f}, Tone: {data['avg_tone']:.2f})\n"
    
    # Calculate overall statistics
    overall_goldstein = trend_data['avg_goldstein'].mean()
    overall_tone = trend_data['avg_tone'].mean()
    total_events = trend_data['country_events'].sum()
    
    trend_summary += f"""
Overall Statistics:
- Total Events: {total_events}
- Average Goldstein Scale: {overall_goldstein:.2f}
- Average Tone: {overall_tone:.2f}
- Unique Countries Involved: {trend_data['country'].nunique()}
"""
    
    print(f"\n🔍 TREND SUMMARY FOR DEEPSEEK:")
    print("="*80)
    print(trend_summary)
    
else:
    print("❌ No trend data available")
    trend_summary = None

## Demo 3: Trend Analysis and Predictions

Let's ask DeepSeek to analyze trends over time and provide predictions about future developments.

In [ ]:
# Ask DeepSeek for country-specific analysis
if country_summary and check_deepseek_server():
    country_prompt = f"""
    As a geopolitical analyst specializing in {target_country}, please analyze these recent events:

    {country_summary}

    Additional context:
    - Average Goldstein Scale: {avg_goldstein:.2f} (range: -10 to +10, where negative = conflict, positive = cooperation)
    - Average Tone: {avg_tone:.2f} (range: -100 to +100, where negative = negative sentiment)

    Please provide:
    1. Assessment of {target_country}'s current geopolitical position based on these events
    2. Key relationships and partnerships evident in the data
    3. Areas of tension or conflict involvement
    4. Dominant types of activities or engagements
    5. Sentiment analysis of {target_country}'s international interactions
    6. Potential implications for foreign policy and international relations
    7. Recommendations for monitoring or areas requiring attention

    Be specific and cite the events in your analysis.
    """
    
    print(f"🤖 Asking DeepSeek to analyze {target_country} events...")
    print("This may take 30-60 seconds...\n")
    
    start_time = time.time()
    response = query_deepseek(country_prompt, max_tokens=2000, temperature=0.3)
    end_time = time.time()
    
    if "error" in response:
        print(f"❌ Error: {response['error']}")
    else:
        country_analysis = response['choices'][0]['message']['content']
        print(f"🔍 DEEPSEEK {target_country} ANALYSIS:")
        print("="*80)
        print(country_analysis)
        print("="*80)
        print(f"\n⏱️ Analysis completed in {end_time - start_time:.1f} seconds")
        
else:
    print("⚠️ Skipping country analysis - either no data or server not available")

In [ ]:
# Focus on a specific country for detailed analysis
def analyze_country_events(country_code, days_back=7):
    """Get events for a specific country"""
    try:
        conn = duckdb.connect(GDELT_DB_PATH)
        
        # Calculate date range
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days_back)
        start_date_str = start_date.strftime("%Y%m%d")
        
        query = """
        SELECT 
            GLOBALEVENTID,
            SQLDATE,
            Actor1Name,
            Actor2Name,
            EventCode,
            GoldsteinScale,
            AvgTone,
            ActionGeo_Fullname,
            ActionGeo_CountryCode
        FROM gdelt_events 
        WHERE (ActionGeo_CountryCode = ? OR Actor1CountryCode = ? OR Actor2CountryCode = ?)
        AND SQLDATE >= ?
        ORDER BY SQLDATE DESC, GoldsteinScale DESC
        LIMIT 15
        """
        
        df = conn.execute(query, [country_code, country_code, country_code, start_date_str]).df()
        conn.close()
        return df
    except Exception as e:
        print(f"Error fetching country data: {e}")
        return pd.DataFrame()

# Choose a country to analyze (let's use US for this example)
target_country = "US"
print(f"🇺🇸 Analyzing recent events involving {target_country}...")

country_events = analyze_country_events(target_country, days_back=14)

if not country_events.empty:
    print(f"✅ Found {len(country_events)} events involving {target_country}")
    
    # Create summary for DeepSeek
    country_summary = f"Recent events involving {target_country} (last 14 days):\n\n"
    
    for i, (_, row) in enumerate(country_events.iterrows(), 1):
        country_summary += f"{i}. {row['Actor1Name']} -> {row['Actor2Name']} "
        country_summary += f"(Event: {row['EventCode']}, Goldstein: {row['GoldsteinScale']}, "
        country_summary += f"Tone: {row['AvgTone']:.1f}, Location: {row['ActionGeo_Fullname']})\n"
    
    print("📊 COUNTRY EVENTS SUMMARY:")
    print("="*60)
    print(country_summary)
    
    # Calculate basic statistics
    avg_goldstein = country_events['GoldsteinScale'].mean()
    avg_tone = country_events['AvgTone'].mean()
    
    print(f"\n📈 STATISTICS:")
    print(f"Average Goldstein Scale: {avg_goldstein:.2f}")
    print(f"Average Tone: {avg_tone:.2f}")
    print(f"Most Common Event Codes: {country_events['EventCode'].value_counts().head(3).to_dict()}")
    
else:
    print(f"❌ No recent events found for {target_country}")
    country_summary = None

## Demo 2: Country-Specific Analysis

Let's focus on a specific country and ask DeepSeek to provide detailed analysis of events involving that country.

In [ ]:
# Ask DeepSeek to analyze the GDELT events
if 'summary_text' in locals() and check_deepseek_server():
    analysis_prompt = f"""
    As a geopolitical analyst, please analyze these recent GDELT events and provide insights:

    {summary_text}

    Please provide:
    1. Key patterns you observe in these events
    2. Notable geopolitical trends or tensions
    3. Countries or regions showing significant activity
    4. Event types that are most prevalent
    5. Overall tone analysis (positive/negative trends)
    6. Any potential implications or connections between events

    Focus on actionable insights that would be valuable for monitoring global events.
    """
    
    print("🤖 Asking DeepSeek to analyze the GDELT events...")
    print("This may take 30-60 seconds...\n")
    
    start_time = time.time()
    response = query_deepseek(analysis_prompt, max_tokens=1500, temperature=0.3)
    end_time = time.time()
    
    if "error" in response:
        print(f"❌ Error: {response['error']}")
    else:
        analysis = response['choices'][0]['message']['content']
        print("🔍 DEEPSEEK GEOPOLITICAL ANALYSIS:")
        print("="*80)
        print(analysis)
        print("="*80)
        print(f"\n⏱️ Analysis completed in {end_time - start_time:.1f} seconds")
        
        # Store the analysis for later use
        deepseek_analysis = analysis
        
else:
    print("⚠️ Skipping DeepSeek analysis - either no data or server not available")

In [ ]:
# Fetch recent GDELT events for analysis
def get_recent_gdelt_sample(limit=50):
    """Get recent GDELT events for analysis"""
    try:
        conn = duckdb.connect(GDELT_DB_PATH)
        
        query = """
        SELECT 
            GLOBALEVENTID,
            SQLDATE,
            Actor1Name,
            Actor2Name,
            EventCode,
            GoldsteinScale,
            AvgTone,
            ActionGeo_CountryCode,
            ActionGeo_Fullname
        FROM gdelt_events 
        WHERE SQLDATE >= 20250101
        AND Actor1Name IS NOT NULL 
        AND Actor2Name IS NOT NULL
        ORDER BY SQLDATE DESC 
        LIMIT ?
        """
        
        df = conn.execute(query, [limit]).df()
        conn.close()
        return df
    except Exception as e:
        print(f"Error fetching GDELT data: {e}")
        return pd.DataFrame()

# Get sample data
print("Fetching recent GDELT events...")
gdelt_sample = get_recent_gdelt_sample(20)

if not gdelt_sample.empty:
    print(f"✅ Fetched {len(gdelt_sample)} recent events")
    print("\nSample of recent events:")
    display(gdelt_sample.head())
    
    # Prepare summary for DeepSeek analysis
    event_summary = []
    for _, row in gdelt_sample.head(10).iterrows():
        event_summary.append({
            'actors': f"{row['Actor1Name']} -> {row['Actor2Name']}",
            'event_code': row['EventCode'],
            'goldstein': row['GoldsteinScale'],
            'tone': row['AvgTone'],
            'location': row['ActionGeo_Fullname'],
            'country': row['ActionGeo_CountryCode']
        })
    
    # Convert to readable format for DeepSeek
    summary_text = "Recent GDELT Events Summary:\n\n"
    for i, event in enumerate(event_summary, 1):
        summary_text += f"{i}. {event['actors']} (Event: {event['event_code']}, "
        summary_text += f"Goldstein: {event['goldstein']}, Tone: {event['tone']:.1f}, "
        summary_text += f"Location: {event['location']})\n"
    
    print("\n" + "="*60)
    print("PREPARED SUMMARY FOR DEEPSEEK ANALYSIS:")
    print("="*60)
    print(summary_text)
    
else:
    print("❌ No GDELT data available for analysis")

## Demo 1: GDELT Data Overview Analysis

Let's ask DeepSeek to analyze a sample of GDELT data and provide insights about global events patterns.

In [ ]:
# Import required libraries
import requests
import json
import pandas as pd
import duckdb
import time
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

print("Setting up DeepSeek + GDELT Integration...")

# Configuration
OPENLLM_URL = "http://localhost:3000"
GDELT_DB_PATH = "../etl/duckdatabase"

# Check if we can connect to DeepSeek server
def check_deepseek_server():
    """Check if DeepSeek OpenLLM server is running"""
    try:
        response = requests.get(f"{OPENLLM_URL}/v1/models", timeout=5)
        return response.status_code == 200
    except:
        return False

def query_deepseek(prompt, max_tokens=1000, temperature=0.7):
    """Send a query to the DeepSeek model"""
    url = f"{OPENLLM_URL}/v1/chat/completions"
    
    payload = {
        "model": "deepseek:r1-llama3.1-8b",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": temperature
    }
    
    try:
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": f"Request failed: {e}"}

# Test DeepSeek connection
if check_deepseek_server():
    print("✅ DeepSeek server is running!")
else:
    print("❌ DeepSeek server not accessible. Please start with:")
    print("   poetry run openllm serve deepseek:r1-llama3.1-8b --port 3000")

# Test GDELT database connection
def test_gdelt_connection():
    """Test connection to GDELT database"""
    try:
        conn = duckdb.connect(GDELT_DB_PATH)
        result = conn.execute("SELECT COUNT(*) FROM gdelt_events").fetchone()
        event_count = result[0] if result else 0
        conn.close()
        return event_count
    except Exception as e:
        print(f"GDELT DB Error: {e}")
        return 0

event_count = test_gdelt_connection()
if event_count > 0:
    print(f"✅ GDELT database connected! Found {event_count:,} events")
else:
    print("❌ GDELT database not accessible. Check path:", GDELT_DB_PATH)

# DeepSeek R1 + GDELT Integration Demo

This notebook demonstrates how to use the DeepSeek R1 model with GDELT data for advanced analysis and insights. We'll combine the power of the DeepSeek R1 reasoning model with real GDELT events data to generate meaningful analysis.

## Overview

- **DeepSeek R1**: Advanced reasoning model for complex analysis
- **GDELT Data**: Global events, language, and tone data
- **Integration**: Using AI to analyze global events patterns and generate insights

## Prerequisites

1. OpenLLM server running with DeepSeek R1 model:
   ```bash
   poetry run openllm serve deepseek:r1-llama3.1-8b --port 3000
   ```

2. GDELT database available (DuckDB format)

Let's get started!